# Sigle Model

In [ ]:

import re
import html
import random
import unicodedata
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import f1_score, classification_report
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW
from tqdm import tqdm
import os

# =============================
# 1️⃣ Reproducibility
# =============================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# =============================
# 2️⃣ Arabic Preprocessing
# =============================
ARABIC_DIACRITICS = re.compile(r'[\u0617-\u061A\u064B-\u0652]')

def preprocess_arabic(text):

    if text is None:
        return ""

    text = str(text)
    text = html.unescape(text)
    text = text.lower()

    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'@\w+', ' ', text)
    text = re.sub(r'#', '', text)
    text = re.sub(r'<[^>]+>', ' ', text)

    text = ARABIC_DIACRITICS.sub('', text)
    text = text.replace('ـ', '')

    text = re.sub(r'[إأآا]', 'ا', text)
    text = re.sub(r'[يى]', 'ي', text)
    text = re.sub(r'[ؤ]', 'و', text)
    text = re.sub(r'[ئ]', 'ي', text)

    text = ''.join(ch for ch in text if unicodedata.category(ch)[0] != 'C')

    text = re.sub(r'[^\w\s\u0600-\u06FF]', ' ', text)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# =============================
# 3️⃣ Load Fixed Splits
# =============================
train_df = pd.read_csv("Alldialect_Train_set.csv")
val_df   = pd.read_csv("Alldialect_Val_set.csv")
test_df  = pd.read_csv("End-to-End_dataset_test.csv")

train_df.columns = train_df.columns.str.strip()
val_df.columns   = val_df.columns.str.strip()
test_df.columns  = test_df.columns.str.strip()

for df, name in [(train_df,"train"),(val_df,"val"),(test_df,"test")]:

    print(f"--- {name.upper()} DataFrame Columns ---")
    print(df.columns)

    if "commentaire" not in df.columns or "classe" not in df.columns:
        raise ValueError(f"{name}.csv must contain columns 'commentaire' and 'classe'")

# =============================
# 4️⃣ Label Encoding
# =============================
label_mapping = {"abusive": 0, "hate": 1, "normal": 2}
label_names = ["abusive", "hate", "normal"]
num_labels = len(label_names)

for df in [train_df, val_df, test_df]:

    df["classe"] = df["classe"].astype(str).str.strip().str.lower()
    df["label"] = df["classe"].map(label_mapping)

    if df["label"].isnull().any():
        raise ValueError(
            "Found invalid labels: "
            f"{set(df['classe'].unique())}"
        )

print("Label mapping:", label_mapping)

print("Label distribution (train):")
print(train_df["classe"].value_counts())

# =============================
# 5️⃣ Apply Preprocessing
# =============================
print("Applying preprocessing...")

train_df["commentaire"] = train_df["commentaire"].map(preprocess_arabic)
val_df["commentaire"]   = val_df["commentaire"].map(preprocess_arabic)
test_df["commentaire"]  = test_df["commentaire"].map(preprocess_arabic)

# =============================
# 6️⃣ Tokenizer
# =============================
MODEL_NAME = "UBC-NLP/MARBERT"

MAX_LEN = 128
BATCH_SIZE = 16

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class HateDataset(Dataset):

    def __init__(self,df):
        self.texts = df["commentaire"].tolist()
        self.labels = df["label"].astype(int).tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self,idx):

        encoding = tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt"
        )

        item = {k:v.squeeze(0) for k,v in encoding.items()}
        item["labels"] = torch.tensor(self.labels[idx],dtype=torch.long)

        return item


train_loader = DataLoader(HateDataset(train_df),batch_size=BATCH_SIZE,shuffle=True)
val_loader   = DataLoader(HateDataset(val_df),batch_size=BATCH_SIZE)
test_loader  = DataLoader(HateDataset(test_df),batch_size=BATCH_SIZE)

# =============================
# 7️⃣ Model
# =============================
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels
)

model.to(device)

# =============================
# 8️⃣ Loss Function
# =============================
criterion = torch.nn.CrossEntropyLoss()

# =============================
# 9️⃣ Optimizer & Scheduler
# =============================
EPOCHS = 5
LR = 2e-5

optimizer = AdamW(model.parameters(),lr=LR)

total_steps = len(train_loader)*EPOCHS

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1*total_steps),
    num_training_steps=total_steps
)

# =============================
# 🔟 Training
# =============================
scaler = torch.cuda.amp.GradScaler()

best_f1 = 0
patience = 2
counter = 0

for epoch in range(EPOCHS):

    model.train()
    total_loss = 0

    for batch in tqdm(train_loader,desc=f"Epoch {epoch+1}"):

        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        with torch.cuda.amp.autocast():

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            loss = criterion(outputs.logits,labels)

        scaler.scale(loss).backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)

        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item()

    print("Train Loss:",total_loss/len(train_loader))

    model.eval()

    preds=[]
    true=[]

    with torch.no_grad():

        for batch in val_loader:

            input_ids=batch["input_ids"].to(device)
            attention_mask=batch["attention_mask"].to(device)
            labels=batch["labels"].to(device)

            outputs=model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            predictions=torch.argmax(outputs.logits,dim=1)

            preds.extend(predictions.cpu().numpy())
            true.extend(labels.cpu().numpy())

    f1=f1_score(true,preds,average="macro")

    print("Validation Macro F1:",f1)

    if f1>best_f1:

        best_f1=f1
        torch.save(model.state_dict(),"best_model.pt")
        counter=0

    else:

        counter+=1

        if counter>=patience:

            print("Early stopping")
            break

# =============================
# 1️⃣1️⃣ Test Evaluation
# =============================
model.load_state_dict(torch.load("best_model.pt"))

model.eval()

preds=[]
true=[]

with torch.no_grad():

    for batch in test_loader:

        input_ids=batch["input_ids"].to(device)
        attention_mask=batch["attention_mask"].to(device)
        labels=batch["labels"].to(device)

        outputs=model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        predictions=torch.argmax(outputs.logits,dim=1)

        preds.extend(predictions.cpu().numpy())
        true.extend(labels.cpu().numpy())

print("\nFinal Test Results:")

print(
    classification_report(
        true,
        preds,
        target_names=label_names,
        digits=4
    )
)

print("Macro F1:",f1_score(true,preds,average="macro"))

Using device: cuda
--- TRAIN DataFrame Columns ---
Index(['commentaire', 'dialect', 'classe'], dtype='object')
--- VAL DataFrame Columns ---
Index(['commentaire', 'dialect', 'classe'], dtype='object')
--- TEST DataFrame Columns ---
Index(['sentence_id', 'commentaire', 'dialect', 'classe'], dtype='object')
Label mapping: {'abusive': 0, 'hate': 1, 'normal': 2}
Label distribution (train):
classe
normal     10986
hate        4632
abusive     3312
Name: count, dtype: int64
Applying preprocessing...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/376 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

Train Loss: 0.4035284964216722
Validation Macro F1: 0.7254114957917049


Epoch 4:   0%|          | 0/1184 [00:00<?, ?it/s]/tmp/ipykernel_757/1442829425.py:213: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 4: 100%|██████████| 1184/1184 [03:06<00:00,  6.35it/s]


Train Loss: 0.3013549630303641
Validation Macro F1: 0.7329016192526993


Epoch 5:   0%|          | 0/1184 [00:00<?, ?it/s]/tmp/ipykernel_757/1442829425.py:213: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 5: 100%|██████████| 1184/1184 [03:06<00:00,  6.35it/s]


Train Loss: 0.21492805432628942
Validation Macro F1: 0.7319533799345623

Final Test Results:
              precision    recall  f1-score   support

     abusive     0.5652    0.7222    0.6341       414
        hate     0.7327    0.6759    0.7031       580
      normal     0.8826    0.8376    0.8595      1373

    accuracy                         0.7778      2367
   macro avg     0.7268    0.7452    0.7323      2367
weighted avg     0.7903    0.7778    0.7818      2367

Macro F1: 0.7322590445635399
